# Blinkit Data Analytics — Feedback analysis
DuckDB SQL + Python | Yash Prajapati

## 1. Setup

### 1.1 Libraries

In [1]:
import warnings, math, textwrap
warnings.filterwarnings('ignore')
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
print('duckdb', duckdb.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

duckdb 1.5.5 | pandas 3.0.2 | numpy 2.4.4


### 1.2 Load cleaned workbook

In [2]:
XLSX = 'data/Blinkit_analysis_new.xlsx'
book = pd.read_excel(XLSX, sheet_name=None)
geo = pd.read_csv('data/city_state_zone.csv')
for name, df in book.items():
    print(f'{name:28s} {df.shape[0]:>6,} rows  {df.shape[1]:>3} cols')

Data_Quality_Report              46 rows    3 cols
Orders_Customer_Info          5,000 rows   16 cols
Orders_Raw_Archive            5,000 rows   23 cols
Order_Line_Items              5,000 rows   14 cols
Delivery_Performance          5,000 rows    8 cols
Customer_Feedback             5,000 rows    8 cols
Customers                     2,500 rows   11 cols
Products                        268 rows   10 cols
Inventory_Analysis              268 rows   18 cols
Data_Dictionary                  44 rows    4 cols
Reference_Parameters             11 rows    5 cols
Relational_Diagram                0 rows    0 cols
Pivot_Payment_Method              7 rows    7 cols
Pivot_Customer_Segment            7 rows    5 cols
Pivot_Category_Sales             14 rows    4 cols
Pivot_Monthly_Trend              24 rows    6 cols
Pivot_Area_Delivery              23 rows    6 cols
Pivot_Inventory_Movement          6 rows    7 cols
Pivot_Category_Stock             14 rows    8 cols


### 1.3 Create DuckDB database and load tables

In [3]:
con = duckdb.connect('blinkit.duckdb')
load = {'orders_src':'Orders_Raw_Archive','items_src':'Order_Line_Items','delivery_src':'Delivery_Performance',
        'feedback_src':'Customer_Feedback','customers_src':'Customers','products_src':'Products','inventory_src':'Inventory_Analysis'}
for tbl, sheet in load.items():
    df = book[sheet].copy()
    df.columns = [c.strip() for c in df.columns]
    con.register('tmp_df', df)
    con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df')
con.register('geo_df', geo)
con.execute('CREATE OR REPLACE TABLE geo AS SELECT * FROM geo_df')
con.execute("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name").df()

,table_name,estimated_size
0,customers_src,2500
1,delivery_src,5000
2,feedback_src,5000
3,geo,316
4,inventory_src,268
5,items_src,5000
6,orders_src,5000
7,products_src,268


### 1.4 Typed analysis views

In [4]:
con.execute('''
CREATE OR REPLACE VIEW orders AS
SELECT order_id, customer_id,
       strptime(order_date, '%d-%m-%Y %H:%M')              AS order_ts,
       CAST(strptime(order_date, '%d-%m-%Y %H:%M') AS DATE) AS order_date,
       strptime(promised_delivery_time, '%d-%m-%Y %H:%M')   AS promised_ts,
       strptime(actual_delivery_time,  '%d-%m-%Y %H:%M')    AS actual_ts,
       delivery_status, order_total, payment_method, delivery_partner_id, store_id,
       CAST(delivery_time_minutes AS INTEGER)               AS delay_min,
       distance_km, reasons_if_delayed, customer_name,
       trim(area) AS area, pincode, customer_segment,
       CAST(registration_date AS DATE)                      AS registration_date,
       order_day_of_week, order_time_slot, order_value_segment,
       CASE WHEN is_weekend = 'Yes' THEN 1 ELSE 0 END       AS is_weekend
FROM orders_src ''')

con.execute('''
CREATE OR REPLACE VIEW items AS
SELECT order_id, product_id, quantity, unit_price, product_name, category, brand,
       price, mrp, margin_percentage, shelf_life_days, min_stock_level, max_stock_level, line_total,
       line_total * margin_percentage / 100.0 AS margin_value
FROM items_src ''')

con.execute('''
CREATE OR REPLACE VIEW feedback AS
SELECT feedback_id, order_id, customer_id, rating, feedback_category, sentiment,
       CAST(feedback_date AS DATE) AS feedback_date
FROM feedback_src ''')

con.execute('''
CREATE OR REPLACE VIEW f_sales AS
SELECT o.order_id, o.customer_id, o.order_ts, o.order_date,
       date_trunc('month', o.order_date)  AS order_month,
       extract(hour FROM o.order_ts)      AS order_hour,
       o.order_day_of_week, o.is_weekend, o.order_time_slot, o.order_value_segment,
       o.payment_method, o.customer_segment, o.registration_date, o.customer_name,
       o.area, g.state, g.zone, g.city_tier,
       i.product_id, i.product_name, i.category, i.brand,
       i.quantity, i.line_total AS revenue, i.margin_value, i.margin_percentage,
       i.price, i.mrp, i.shelf_life_days,
       o.delay_min, o.distance_km, o.delivery_status,
       CASE WHEN o.delivery_status = 'On Time' THEN 1 ELSE 0 END AS is_on_time_status,
       CASE WHEN o.delay_min > 0 THEN 1 ELSE 0 END               AS is_late_minutes,
       f.rating, f.sentiment, f.feedback_category
FROM orders o
JOIN items i    ON i.order_id = o.order_id
LEFT JOIN geo g ON g.area     = o.area
LEFT JOIN feedback f ON f.order_id = o.order_id ''')

con.execute('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_date) AS first_day, MAX(order_date) AS last_day FROM f_sales').df()

,fact_rows,orders,first_day,last_day
0,5000,5000,2023-03-16,2024-11-04


### 1.5 Query helper

In [5]:
def q(sql, con=con):
    return con.execute(textwrap.dedent(sql)).df()

def pct(x, n):
    return round(100.0 * x / n, 2) if n else 0.0

q('SELECT COUNT(*) AS rows_in_fact_view FROM f_sales')

,rows_in_fact_view
0,5000


## 11. Customer feedback

### 11.1 Rating distribution

In [6]:
q('''
SELECT rating, COUNT(*) AS reviews,
       round(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct,
       round(AVG(revenue), 0) AS avg_order_value,
       round(AVG(delay_min), 2) AS avg_minutes_vs_promise
FROM f_sales WHERE rating IS NOT NULL GROUP BY rating ORDER BY rating ''')

,rating,reviews,share_pct,avg_order_value,avg_minutes_vs_promise
0,1,540,10.80,988.00,3.83
1,2,538,10.76,973.00,4.97
2,3,1398,27.96,"1,013.00",4.72
3,4,1708,34.16,"1,005.00",4.26
4,5,816,16.32,959.00,4.40


### 11.2 Sentiment by feedback topic

In [7]:
q('''
SELECT feedback_category AS topic, COUNT(*) AS reviews,
       COUNT(*) FILTER (WHERE sentiment = 'Positive') AS positive,
       COUNT(*) FILTER (WHERE sentiment = 'Neutral')  AS neutral,
       COUNT(*) FILTER (WHERE sentiment = 'Negative') AS negative,
       round(100.0 * COUNT(*) FILTER (WHERE sentiment = 'Negative') / COUNT(*), 1) AS negative_pct,
       round(AVG(rating), 2) AS avg_rating
FROM f_sales WHERE feedback_category IS NOT NULL GROUP BY 1 ORDER BY negative_pct DESC ''')

,topic,reviews,positive,neutral,negative,negative_pct,avg_rating
0,Product Quality,1250,378,440,432,34.60,3.32
1,Customer Service,1266,418,431,417,32.90,3.37
2,App Experience,1213,404,421,388,32.00,3.36
3,Delivery,1271,420,446,405,31.90,3.33


### 11.3 Rating by delivery status

In [8]:
q('''
SELECT delivery_status, COUNT(*) AS reviews, round(AVG(rating), 3) AS avg_rating,
       round(stddev_samp(rating), 3) AS sd_rating,
       round(100.0 * COUNT(*) FILTER (WHERE rating <= 2) / COUNT(*), 2) AS low_rating_pct,
       round(100.0 * COUNT(*) FILTER (WHERE sentiment = 'Negative') / COUNT(*), 2) AS negative_pct
FROM f_sales WHERE rating IS NOT NULL GROUP BY 1 ORDER BY avg_rating DESC ''')

,delivery_status,reviews,avg_rating,sd_rating,low_rating_pct,negative_pct
0,Slightly Delayed,1037,3.39,1.14,20.25,32.79
1,On Time,3470,3.33,1.20,21.87,32.68
2,Significantly Delayed,493,3.33,1.19,22.11,34.08


### 11.4 Is the rating difference across delivery statuses real

In [9]:
groups = [g.rating.values for _, g in q('SELECT delivery_status, rating FROM f_sales WHERE rating IS NOT NULL').groupby('delivery_status')]
anova = stats.f_oneway(*groups)
kruskal = stats.kruskal(*groups)
pd.DataFrame([{'test':'One-way ANOVA on rating by delivery status','statistic':round(anova.statistic,4),'p_value':round(anova.pvalue,4)},
              {'test':'Kruskal-Wallis on rating by delivery status','statistic':round(kruskal.statistic,4),'p_value':round(kruskal.pvalue,4)},
              {'test':'Spearman: delay minutes vs rating',
               'statistic':round(stats.spearmanr(*q('SELECT delay_min, rating FROM f_sales WHERE rating IS NOT NULL').T.values).statistic,4),
               'p_value':round(stats.spearmanr(*q('SELECT delay_min, rating FROM f_sales WHERE rating IS NOT NULL').T.values).pvalue,4)}])

,test,statistic,p_value
0,One-way ANOVA on rating by delivery status,1.07,0.34
1,Kruskal-Wallis on rating by delivery status,1.20,0.55
2,Spearman: delay minutes vs rating,-0.01,0.58


### 11.5 Rating by delay band

In [10]:
q('''
SELECT CASE WHEN delay_min <= 0 THEN 'a. on or before promise'
            WHEN delay_min <= 10 THEN 'b. 1-10 minutes late'
            WHEN delay_min <= 20 THEN 'c. 11-20 minutes late'
            ELSE 'd. more than 20 minutes late' END AS delay_band,
       COUNT(*) AS reviews, round(AVG(rating), 3) AS avg_rating,
       round(100.0 * COUNT(*) FILTER (WHERE sentiment = 'Negative') / COUNT(*), 2) AS negative_pct
FROM f_sales WHERE rating IS NOT NULL GROUP BY 1 ORDER BY 1 ''')

,delay_band,reviews,avg_rating,negative_pct
0,a. on or before promise,1902,3.35,32.23
1,b. 1-10 minutes late,2091,3.33,32.76
2,c. 11-20 minutes late,674,3.39,33.83
3,d. more than 20 minutes late,333,3.30,34.83


### 11.6 Rating by category and by zone

In [11]:
q('''
SELECT category, round(AVG(rating), 3) AS avg_rating, COUNT(*) AS reviews,
       round(100.0 * COUNT(*) FILTER (WHERE rating <= 2) / COUNT(*), 2) AS low_rating_pct
FROM f_sales WHERE rating IS NOT NULL GROUP BY 1 ORDER BY avg_rating DESC ''')

,category,avg_rating,reviews,low_rating_pct
0,Grocery & Staples,3.46,449,19.15
1,Household Care,3.40,509,19.84
2,Pharmacy,3.38,481,19.33
3,Pet Care,3.36,501,20.56
4,Fruits & Vegetables,3.36,492,20.73
5,Cold Drinks & Juices,3.34,375,22.13
6,Snacks & Munchies,3.33,483,23.60
7,Instant & Frozen Food,3.31,356,19.94
8,Baby Care,3.30,334,24.25
9,Personal Care,3.29,454,22.91


### 11.7 Repeat behaviour of customers who left a low rating

In [12]:
q('''
WITH low AS (SELECT DISTINCT customer_id FROM f_sales WHERE rating <= 2),
o AS (SELECT customer_id, COUNT(DISTINCT order_id) AS orders FROM f_sales GROUP BY 1)
SELECT CASE WHEN l.customer_id IS NULL THEN 'no low rating given' ELSE 'gave a rating of 1 or 2' END AS customer_group,
       COUNT(*) AS customers, round(AVG(o.orders), 2) AS avg_orders,
       round(100.0 * COUNT(*) FILTER (WHERE o.orders >= 2) / COUNT(*), 2) AS repeat_rate_pct
FROM o LEFT JOIN low l ON l.customer_id = o.customer_id GROUP BY 1 ''')

,customer_group,customers,avg_orders,repeat_rate_pct
0,gave a rating of 1 or 2,872,2.79,83.14
1,no low rating given,1300,1.97,59.00
